# NB1 — Preparação
**Carrega dados brutos, constrói séries temporais, gera features, índice global.**  
Rodar **uma vez**. Só re-rodar se mudar dados brutos ou adicionar portos.

### Outputs → `data/processed/`
| Arquivo | Conteúdo |
|---------|----------|
| `series_diario.parquet` | Séries diárias (porto × date × 4 dims) |
| `series_semanal.parquet` | Séries semanais |
| `features_semanal.parquet` | Séries + features temporais + calendário |
| `indice_global.parquet` | Índice PortWatch semanal (gi_portcalls, gi_z) |
| `portos_clusters.csv` | Portos PortWatch com cluster, PCA, features |
| `mapeamento_portos.csv` | ANTAQ ↔ PortWatch |

In [ ]:
# Célula 0 — Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from statsmodels.tsa.seasonal import STL
from scipy import stats
from pathlib import Path
import re as re_mod
import warnings
warnings.filterwarnings("ignore")

import sys; sys.path.insert(0, str(Path.cwd().parent / "src"))
from config import *

plt.rcParams.update({"figure.figsize": (14, 5), "figure.dpi": 100})
print("Config OK —", ROOT)

In [ ]:
# Célula 1 — Carregar ANTAQ (Atracação + Tempos + Carga)
# v9: leitura direta de zips anuais (artigo/data/raw/antaq/)
# Fallback: .txt extraídos em dados/estatistico/ (legado)

import zipfile, io

def load_antaq_yearly(file_suffix, zip_dir=ANTAQ_RAW_ZIPS, fallback_dir=ANTAQ_RAW_REAL):
    """Carrega todos os arquivos ANTAQ de um tipo (ex: 'Atracacao.txt').
    
    Tenta primeiro ler dos zips anuais (2010.zip … 2026.zip),
    depois volta ao diretório de .txt extraídos (legado).
    
    Args:
        file_suffix: Nome do arquivo dentro do zip, sem o prefixo de ano.
                     Ex: 'Atracacao.txt' → procura '2010Atracacao.txt' em 2010.zip
        zip_dir:     Diretório com os zips anuais.
        fallback_dir: Diretório com os .txt já extraídos (legado).
    """
    dfs = []
    
    # ── Tentativa 1: ler dos zips anuais ──
    zip_files = sorted(zip_dir.glob("[0-9]*.zip")) if zip_dir.exists() else []
    if zip_files:
        for zf_path in zip_files:
            year = zf_path.stem  # ex: "2024"
            target = f"{year}{file_suffix}"
            try:
                with zipfile.ZipFile(zf_path) as zf:
                    # Buscar o arquivo dentro do zip (pode ter subpasta)
                    candidates = [n for n in zf.namelist() if n.endswith(target) or n == target]
                    if not candidates:
                        continue
                    with zf.open(candidates[0]) as f:
                        raw_bytes = f.read()
                        # Detectar encoding: tentar utf-8-sig, depois latin-1
                        for enc in [ANTAQ_ENCODING, "latin-1"]:
                            try:
                                text = raw_bytes.decode(enc)
                                break
                            except UnicodeDecodeError:
                                continue
                        df = pd.read_csv(
                            io.StringIO(text), sep=ANTAQ_SEP, low_memory=False
                        )
                        dfs.append(df)
            except Exception as e:
                print(f"  AVISO: {zf_path.name}/{target} — {e}")
        
        if dfs:
            print(f"  → Lidos {len(dfs)} zips para *{file_suffix}")
            return pd.concat(dfs, ignore_index=True)
    
    # ── Tentativa 2: fallback para .txt extraídos ──
    # Fix: usar regex exato para evitar que *Atracacao.txt capture
    # TemposAtracacao.txt, TaxaOcupacaoTOAtracacao.txt, etc.
    if fallback_dir.exists():
        all_files = sorted(fallback_dir.glob(f"*{file_suffix}"))
        all_files = [f for f in all_files
                     if re_mod.match(rf'^\d{{4}}{re_mod.escape(file_suffix)}$', f.name)]
        for f in all_files:
            try:
                df = pd.read_csv(f, sep=ANTAQ_SEP, encoding=ANTAQ_ENCODING, low_memory=False)
                dfs.append(df)
            except Exception as e:
                print(f"  ERRO fallback: {f.name} — {e}")
        if dfs:
            print(f"  → Lidos {len(dfs)} .txt (fallback) para *{file_suffix}")
            return pd.concat(dfs, ignore_index=True)
    
    raise ValueError(f"Nenhum arquivo encontrado para *{file_suffix} "
                     f"(zip_dir={zip_dir}, fallback={fallback_dir})")

# --- Carregar as 3 tabelas ---
print("Carregando Atracacao...")
raw_atrac = load_antaq_yearly("Atracacao.txt")
print(f"  {len(raw_atrac):,} registros")

print("Carregando TemposAtracacao...")
raw_tempos = load_antaq_yearly("TemposAtracacao.txt")
print(f"  {len(raw_tempos):,} registros")

print("Carregando Carga...")
raw_carga = load_antaq_yearly("Carga.txt")
print(f"  {len(raw_carga):,} registros")

# --- Preparar campos numéricos (podem conter "Valor Discrepante" ou vírgula decimal) ---
# T1 horas
raw_tempos["t1_horas"] = pd.to_numeric(
    raw_tempos[TEMPOS_T1_COL].astype(str).str.replace(",", "."),
    errors="coerce"
)
# TAtracado horas (v8: nova 4ª dimensão — tempo total no berço = T2+T3+T4)
raw_tempos["tatracado_horas"] = pd.to_numeric(
    raw_tempos[TEMPOS_TATRACADO_COL].astype(str).str.replace(",", "."),
    errors="coerce"
)

# Tonelagem
raw_carga["tonelagem"] = pd.to_numeric(
    raw_carga[CARGA_TONELAGEM_COL].astype(str).str.replace(",", "."),
    errors="coerce"
)

# --- Merge ---
raw = raw_atrac[[ANTAQ_ID_COL, ANTAQ_PORT_COL, ANTAQ_DATE_COL,
                  ANTAQ_NAV_COL]].copy()
raw = raw.merge(raw_tempos[[ANTAQ_ID_COL, "t1_horas", "tatracado_horas"]],
                on=ANTAQ_ID_COL, how="left")

# Carga: agregar por IDAtracacao (pode ter múltiplas linhas de carga por atracação)
carga_agg = raw_carga.groupby([ANTAQ_ID_COL, CARGA_SENTIDO_COL])["tonelagem"].sum().reset_index()
# Tonelagem exportação
carga_exp = carga_agg[carga_agg[CARGA_SENTIDO_COL].isin(EXPORT_VALUES)]
carga_exp = carga_exp.groupby(ANTAQ_ID_COL)["tonelagem"].sum().reset_index()
carga_exp.columns = [ANTAQ_ID_COL, "tonelagem_exp"]
raw = raw.merge(carga_exp, on=ANTAQ_ID_COL, how="left")
raw["tonelagem_exp"] = raw["tonelagem_exp"].fillna(0)

# Renomear
raw = raw.rename(columns={
    ANTAQ_ID_COL: "id_atracacao",
    ANTAQ_PORT_COL: "porto_complexo",
    ANTAQ_DATE_COL: "data_atracacao",
    ANTAQ_NAV_COL: "tipo_navegacao",
})
raw["data_atracacao"] = pd.to_datetime(raw["data_atracacao"], dayfirst=True, errors="coerce")

# Mapear nome curto
raw["porto"] = raw["porto_complexo"].map(PORTO_NAMES)

print(f"\nANTAQ merged: {len(raw):,} registros")
print(f"Portos com mapeamento: {raw['porto'].notna().sum():,} / {len(raw):,}")
print(f"Período: {raw['data_atracacao'].min().date()} — {raw['data_atracacao'].max().date()}")
print(f"\nNavegação:")
print(raw["tipo_navegacao"].value_counts())
print(f"\nTop 20 por Complexo Portuário:")
print(raw.groupby("porto_complexo")["id_atracacao"].count().sort_values(ascending=False).head(20))

In [ ]:
# Célula 2 — Filtro Longo Curso + Selecionar portos + Limpar

# Ranking por nomes mapeados (só portos com mapeamento)
raw_mapped = raw[raw["porto"].notna()].copy()

# ── v8: Filtro Longo Curso ────────────────────────────────────────────
# Dados filtrados para navegação de longo curso, excluindo cabotagem,
# apoio e navegação interior — consistente com o escopo de monitoramento
# de disrupções no comércio exterior (PortWatch, ComexStat).
n_pre = len(raw_mapped)
raw_mapped = raw_mapped[raw_mapped["tipo_navegacao"] == ANTAQ_NAV_LONGO_CURSO].copy()
n_pos = len(raw_mapped)
print(f"Filtro Longo Curso: {n_pre:,} → {n_pos:,} registros "
      f"({n_pos/n_pre*100:.1f}% retidos)")

ranking = raw_mapped.groupby("porto")["id_atracacao"].count().sort_values(ascending=False)
print("\nRanking de portos (Longo Curso):")
print(ranking.head(20))

portos = ranking.head(TOP_N_PORTOS).index.tolist()
df = raw_mapped[raw_mapped["porto"].isin(portos)].copy()

# Limpeza
df.loc[df["t1_horas"] < 0, "t1_horas"] = np.nan
df.loc[df["t1_horas"] > T1_OUTLIER_MAX_HORAS, "t1_horas"] = np.nan
df.loc[df["tatracado_horas"] < 0, "tatracado_horas"] = np.nan
df.loc[df["tatracado_horas"] > T1_OUTLIER_MAX_HORAS, "tatracado_horas"] = np.nan
df.loc[df["tonelagem_exp"] < 0, "tonelagem_exp"] = 0

print(f"\nSelecionados: {len(portos)} portos, {len(df):,} registros")
print(f"Portos: {portos}")

# Cobertura semanal rápida
df["_week"] = df["data_atracacao"].dt.to_period("W")
cov = df.groupby("porto")["_week"].nunique().sort_values()
print(f"\nCobertura semanal (semanas com ≥1 atracação LC):")
for p in cov.index:
    print(f"  {p:25s}: {cov[p]} semanas")
df.drop(columns=["_week"], inplace=True)

In [ ]:
# Célula 3 — Séries diárias

df["date"] = df["data_atracacao"].dt.normalize()

# ── Fix 2: cortar período de treino ──────────────────────────────────
# Com dados desde 2000, o modelo absorve tendências de longo prazo e
# perde eventos como a Greve dos Caminhoneiros. 10 anos é suficiente.
cutoff = pd.Timestamp(f"{MIN_DATA_YEAR}-01-01")
df = df[df["date"] >= cutoff].copy()
print(f"Fix 2 — Dados a partir de {MIN_DATA_YEAR}: {len(df):,} registros")

daily_parts = []
for porto in portos:
    dp = df[df["porto"] == porto]

    # Métricas operacionais (v8: tatracado_mediano em vez de pct_paralisacao)
    ops = (dp.groupby("date")
        .agg(atracacoes=("id_atracacao", "count"),
             t1_mediano=("t1_horas", "median"),
             tatracado_mediano=("tatracado_horas", "median"))
        .reset_index())

    # Tonelagem exportação
    ton = dp.groupby("date").agg(tonelagem_exp=("tonelagem_exp", "sum")).reset_index()

    daily = ops.merge(ton, on="date", how="left")
    daily["tonelagem_exp"] = daily["tonelagem_exp"].fillna(0)
    daily["porto"] = porto
    daily_parts.append(daily)

daily = pd.concat(daily_parts, ignore_index=True)

# Preencher gaps
full_range = pd.date_range(daily["date"].min(), daily["date"].max(), freq="D")
filled = []
for porto in portos:
    dp = daily[daily["porto"] == porto].set_index("date").reindex(full_range)
    dp["porto"] = porto
    dp["atracacoes"] = dp["atracacoes"].fillna(0).astype(int)
    dp["tonelagem_exp"] = dp["tonelagem_exp"].fillna(0)
    # T1 e tatracado: NaN em dias sem operação (correto)
    filled.append(dp.reset_index().rename(columns={"index": "date"}))

daily = pd.concat(filled, ignore_index=True)
daily.to_parquet(DATA_PROC / "series_diario.parquet", index=False)
print(f"Séries diárias: {len(daily):,}")
print(f"Período: {daily['date'].min().date()} — {daily['date'].max().date()}")

In [ ]:
# Célula 4 — Séries semanais

daily["yr"] = daily["date"].dt.isocalendar().year.astype(int)
daily["wk"] = daily["date"].dt.isocalendar().week.astype(int)

weekly = (daily.groupby(["porto", "yr", "wk"])
    .agg(atracacoes=("atracacoes", "sum"),
         tonelagem_exp=("tonelagem_exp", "sum"),
         t1_mediano=("t1_mediano", "median"),
         tatracado_mediano=("tatracado_mediano", "median"),
         n_days=("date", "count"))
    .reset_index())

weekly["date"] = pd.to_datetime(
    weekly["yr"].astype(str) + weekly["wk"].astype(str).str.zfill(2) + "1",
    format="%G%V%u")
weekly = weekly[weekly["n_days"] >= 5].copy()

# Descartar última semana de cada porto se incompleta (< 7 dias) — dados ANTAQ
# frequentemente ainda não estão consolidados no final da série
for porto in weekly["porto"].unique():
    mask = weekly["porto"] == porto
    idx_last = weekly.loc[mask, "date"].idxmax()
    if weekly.loc[idx_last, "n_days"] < 7:
        weekly = weekly.drop(idx_last)

weekly = weekly.drop(columns=["yr", "wk", "n_days"])
weekly.to_parquet(DATA_PROC / "series_semanal.parquet", index=False)

for p in portos[:5]:
    print(f"{p}: {(weekly['porto']==p).sum()} semanas")

In [ ]:
# Célula 5 — Features temporais

def add_features(df, col):
    """Features temporais para uma dimensão-alvo."""
    y = df[col]
    for w in MM_WINDOWS:
        df[f"{col}_mm{w}"] = y.rolling(w, min_periods=max(1, w // 2)).mean()
    for lag in LAG_WEEKS:
        df[f"{col}_lag{lag}"] = y.shift(lag)
    for lag in [1, 4, 52]:
        s = y.shift(lag)
        df[f"{col}_pct{lag}w"] = (y - s) / s.replace(0, np.nan)
    mm = y.rolling(ROLLING_STD_WINDOW, min_periods=4).mean()
    df[f"{col}_dev_mm{ROLLING_STD_WINDOW}"] = y - mm
    df[f"{col}_std_r{ROLLING_STD_WINDOW}"] = y.rolling(ROLLING_STD_WINDOW, min_periods=4).std()
    return df

def add_calendar(df):
    df["month_sin"] = np.sin(2 * np.pi * df["date"].dt.month / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["date"].dt.month / 12)
    df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
    return df

parts = []
for porto in weekly["porto"].unique():
    p = weekly[weekly["porto"] == porto].sort_values("date").copy()
    for dim in DIMS:
        p = add_features(p, dim)
    p = add_calendar(p)
    parts.append(p)

feat = pd.concat(parts, ignore_index=True).replace([np.inf, -np.inf], np.nan)
feat.to_parquet(DATA_PROC / "features_semanal.parquet", index=False)
print(f"Features: {feat.shape}")
print(f"Colunas: {sorted(feat.columns.tolist())}")

In [ ]:
# Célula 6 — STL (só para dims principais)

for porto in feat["porto"].unique():
    mask = feat["porto"] == porto
    for dim in STL_DIMS:
        s = feat.loc[mask, dim].interpolate().bfill().ffill()
        if s.notna().sum() < 2 * STL_PERIOD:
            continue
        try:
            r = STL(s, period=STL_PERIOD, robust=True).fit()
            feat.loc[mask, f"{dim}_stl_trend"] = r.trend.values
            feat.loc[mask, f"{dim}_stl_seasonal"] = r.seasonal.values
            feat.loc[mask, f"{dim}_stl_resid"] = r.resid.values
        except:
            pass

feat.to_parquet(DATA_PROC / "features_semanal.parquet", index=False)
print("STL concluído. Features atualizadas.")

In [ ]:
# Célula 7 — Detectar quebras estruturais
# Fix 5: comparar últimas 52 semanas vs 53-104 semanas antes
#         (não vs todo o histórico, que confunde crescimento com quebra).

for porto in portos:
    mask = feat["porto"] == porto
    s = feat.loc[mask, "atracacoes"].dropna()
    if len(s) < 104:
        continue

    recent = s.tail(52)
    prior = s.iloc[-104:-52]     # Fix 5: janela imediatamente anterior
    t, p = stats.ttest_ind(recent, prior)
    ratio = recent.mean() / max(prior.mean(), 1)

    flag = "⚠️ QUEBRA" if (p < 0.001 and abs(ratio - 1) > 0.3) else "  OK"
    print(f"{flag} {porto}: ratio={ratio:.2f}, p={p:.4f} (últimas 52 vs 53-104)")

In [ ]:
# Célula 8 — Carregar PortWatch + clustering

pw = pd.read_csv(PORTWATCH_FILE, parse_dates=["date"])
pw["date"] = pd.to_datetime(pw["date"], utc=True).dt.tz_localize(None)

print(f"PortWatch: {pw.shape}")
print(f"Portos: {pw['portid'].nunique()}, Países: {pw['country'].nunique()}")
print(f"Período: {pw['date'].min().date()} — {pw['date'].max().date()}")
print(f"BR: {pw[pw['country']=='Brazil']['portid'].nunique()} portos")

# Features de clustering
def compute_port_features(pw):
    feats = []
    for pid, g in pw.groupby("portid"):
        total = g["portcalls"]
        m = total.mean()
        if m < 0.5 or len(g) < 365:
            continue
        total_sum = max(total.sum(), 1)
        feats.append({
            "portid": pid,
            "portname": g["portname"].iloc[0],
            "country": g["country"].iloc[0],
            "media_portcalls": m,
            "cv": total.std() / m if m > 0 else 0,
            "sazonalidade_semanal": (
                g.assign(dow=g["date"].dt.dayofweek)
                .groupby("dow")["portcalls"].mean()
                .pipe(lambda x: (x.max() - x.min()) / m if m > 0 else 0)
            ),
            "pct_container": g["portcalls_container"].sum() / total_sum,
            "pct_dry_bulk": g["portcalls_dry_bulk"].sum() / total_sum,
            "pct_tanker": g["portcalls_tanker"].sum() / total_sum,
            "n_days": len(g),
        })
    return pd.DataFrame(feats)

pf = compute_port_features(pw)
print(f"\nPortos após filtro: {len(pf)} (BR: {(pf['country']=='Brazil').sum()})")

# K-Means
X_cl = pf[CLUSTERING_FEATURES].fillna(pf[CLUSTERING_FEATURES].median())
X_sc = StandardScaler().fit_transform(X_cl)

# Elbow + Silhouette
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_sc)
    axes[0].scatter(k, km.inertia_, c="steelblue", s=60)
    axes[1].scatter(k, silhouette_score(X_sc, labels), c="steelblue", s=60)
axes[0].set_title("Elbow"); axes[1].set_title("Silhouette")
plt.tight_layout(); plt.savefig(FIGS / "elbow_silhouette.png", dpi=150); plt.show()

km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=20)
pf["cluster"] = km.fit_predict(X_sc)
pca = PCA(n_components=2).fit(X_sc)
pf["pca1"], pf["pca2"] = pca.transform(X_sc).T

# Visualizar
fig, ax = plt.subplots(figsize=(12, 8))
is_br = pf["country"] == "Brazil"
ax.scatter(pf.loc[~is_br, "pca1"], pf.loc[~is_br, "pca2"],
           c=pf.loc[~is_br, "cluster"], cmap="Set2", alpha=0.3, s=20)
ax.scatter(pf.loc[is_br, "pca1"], pf.loc[is_br, "pca2"],
           c=pf.loc[is_br, "cluster"], cmap="Set2", edgecolors="red",
           linewidths=2, s=100, zorder=5)
for _, r in pf[is_br].iterrows():
    ax.annotate(r["portname"], (r["pca1"], r["pca2"]), fontsize=7)
ax.set_title(f"Clusters globais (K={N_CLUSTERS})")
plt.savefig(FIGS / "pca_clusters.png", dpi=150, bbox_inches="tight"); plt.show()

pf.to_csv(DATA_PROC / "portos_clusters.csv", index=False)
print(f"\nClusters: {pf['cluster'].value_counts().to_dict()}")
print(f"\nBR por cluster:")
print(pf[is_br][["portid", "portname", "cluster", "media_portcalls"]].sort_values("cluster").to_string(index=False))

In [ ]:
# Célula 9 — Índice global semanal
# TRACER: Índice global é mais útil que controles sintéticos.

non_br = pf[pf["country"] != "Brazil"]
top50 = non_br.nlargest(TOP_N_GLOBAL_INDEX, "media_portcalls")["portid"].tolist()

pw_g = pw[pw["portid"].isin(top50)].copy()
pw_g["yr"] = pw_g["date"].dt.isocalendar().year.astype(int)
pw_g["wk"] = pw_g["date"].dt.isocalendar().week.astype(int)

gi = pw_g.groupby(["yr", "wk"]).agg(
    gi_portcalls=("portcalls", "mean"),
    n_dias=("date", "nunique"),
).reset_index()
gi["date"] = pd.to_datetime(
    gi["yr"].astype(str) + gi["wk"].astype(str).str.zfill(2) + "1",
    format="%G%V%u")
gi = gi.sort_values("date")
# Descartar última semana se incompleta (< 5 dias) — PortWatch
if len(gi) > 0 and gi.iloc[-1]["n_dias"] < 5:
    gi = gi.iloc[:-1].copy()
gi["gi_mm12"] = gi["gi_portcalls"].rolling(12, min_periods=6).mean()
gi["gi_std12"] = gi["gi_portcalls"].rolling(12, min_periods=6).std()
gi["gi_z"] = (gi["gi_portcalls"] - gi["gi_mm12"]) / gi["gi_std12"].replace(0, np.nan)

gi.to_parquet(DATA_PROC / "indice_global.parquet", index=False)

# Visualizar
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
axes[0].plot(gi["date"], gi["gi_portcalls"], color="steelblue")
axes[0].set_ylabel("Portcalls médios/dia"); axes[0].set_title("Índice Global (top 50, excl. BR)")
axes[1].plot(gi["date"], gi["gi_z"], color="steelblue")
axes[1].axhline(SCORE_B_THRESHOLD, ls="--", color="red", alpha=0.5)
axes[1].axhline(-SCORE_B_THRESHOLD, ls="--", color="red", alpha=0.5)
axes[1].set_ylabel("Z-score")
for ax in axes:
    for ev in KNOWN_EVENTS:
        if "start" in ev:
            ax.axvspan(ev["start"], ev["end"], alpha=0.1, color="orange")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS / "indice_global.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"Índice global: {len(gi)} semanas ({gi['date'].min().date()} — {gi['date'].max().date()})")

In [ ]:
# Célula 10 — ComexStat (placeholder — quando disponível)
# VERIFICAR: formato e colunas do ComexStat.
# comex = pd.read_csv(COMEXSTAT_RAW / "exp_completa.csv", sep=";")
# comex_agg = comex.groupby([...]).agg(usd=("vl_fob","sum"), kg=("kg_liquido","sum"))
# comex_agg.to_parquet(DATA_PROC / "comex_export.parquet", index=False)
print("ComexStat: PENDENTE — dados não carregados ainda.")

In [ ]:
# Célula 11 — Mapeamento ANTAQ ↔ PortWatch
# Usar mapeamento manual do config.py (mais confiável que fuzzy match)

pw_br = pf[pf["country"] == "Brazil"][["portid", "portname"]].copy()
print("Portos BR no PortWatch:")
print(pw_br.to_string(index=False))

mapa = []
for porto_short, pw_id in ANTAQ_TO_PORTWATCH.items():
    pw_row = pw_br[pw_br["portid"] == pw_id]
    pw_name = pw_row["portname"].values[0] if len(pw_row) > 0 else "NÃO ENCONTRADO"
    mapa.append({
        "porto_antaq": porto_short,
        "portid_portwatch": pw_id,
        "portname_portwatch": pw_name,
        "no_ranking": porto_short in portos,
    })

mapa_df = pd.DataFrame(mapa)
mapa_df.to_csv(DATA_PROC / "mapeamento_portos.csv", index=False)
print(f"\nMapeamento ({len(mapa_df)} portos):")
print(mapa_df.to_string(index=False))

In [ ]:
# Célula 12 — EDA final

top5 = feat.groupby("porto")["atracacoes"].sum().nlargest(5).index.tolist()
fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)
for i, (dim, label) in enumerate(DIM_LABELS.items()):
    ax = axes[i]
    for p in top5:
        m = feat["porto"] == p
        ax.plot(feat.loc[m, "date"], feat.loc[m, dim], label=p, alpha=0.7)
    ax.set_ylabel(label); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    for ev in KNOWN_EVENTS:
        if "start" in ev:
            ax.axvspan(ev["start"], ev["end"], alpha=0.08, color="red")
axes[0].set_title("Séries semanais — Top 5 portos")
plt.tight_layout()
plt.savefig(FIGS / "eda_series.png", dpi=150, bbox_inches="tight"); plt.show()

print("\n✅ NB1 concluído. Outputs salvos em:", DATA_PROC)
print("Arquivos:")
for f in sorted(DATA_PROC.glob("*")):
    if f.is_file():
        print(f"  {f.name} ({f.stat().st_size / 1e6:.1f} MB)")